# CS229 L12 — Foundation Models and Representation Learning

**Video:** Spring 2026 · [YouTube](https://www.youtube.com/watch?v=_kREM2UAiJ8)  
**Instructor:** Tengyu Ma  
**Topics:** Diffusion training recap, foundation model paradigm, representation learning, linear probing, fine-tuning, LPFT, LoRA

---

| Section | Content |
|---|---|
| 1 | Diffusion recap — noise-prediction loss + sampling algorithm |
| 2 | Foundation model paradigm — pre-training + adaptation |
| 3 | Representation learning — $f_\theta: x \to \mathbb{R}^m$ |
| 4 | Linear probing — freeze backbone, train linear head |
| 5 | Fine-tuning — warm initialization, multiple global minima |
| 6 | LPFT — linear probe then fine-tune |
| 7 | LoRA — low-rank adaptation, math, use cases |

## 1. Diffusion Recap — Final Training and Sampling

### Training algorithm (DDPM)

The final simplified training loop:

```
repeat:
  x_0  ← sample from training set
  t    ← Uniform{1, ..., T}
  ε    ← N(0, I)                           # noise to add
  x_t  = √ᾱ_t · x_0 + √(1-ᾱ_t) · ε      # one-shot corrupt
  loss = ||ε - ε_θ(x_t, t)||²             # predict the noise
  θ   ← θ - η · ∇_θ loss
```

### Sampling algorithm

```
x_T ← N(0, I)                              # start from pure noise
for t = T, T-1, ..., 1:
  ε̂  = ε_θ(x_t, t)                        # predict noise with network
  μ̂  = (1/√α_t) · (x_t − β_t/√(1−ᾱ_t) · ε̂)   # denoised mean
  z  ← N(0, I)  if t > 1  else  0
  x_{t-1} = μ̂ + √β̃_t · z                # sample with noise
return x_0
```

**Why it works:**
- Fixed forward process avoids the encoder training instability of VAEs
- Stretching to $T$ steps makes each denoising step a local, easy prediction
- Early steps recover coarse structure (contours); late steps add fine detail

### Why μ̃_t and L1 have the same form

The $t=1$ boundary term $L_1 = \mathbb{E}[\log p_\theta(x_0|x_1)]$ looks different but collapses to the same formula because:
$$x_0 = \frac{x_1 - \sqrt{\beta_1}\,\varepsilon_1}{\sqrt{\alpha_1}} = \tilde{\mu}_1(x_1, x_0)$$
so predicting $x_0$ from $x_1$ has the same noise-matching structure as all other steps.

## 2. Foundation Model Paradigm

The term "foundation model" was coined by Stanford SAIL (Percy Liang et al., 2021) after GPT-3 demonstrated that a single large pre-trained model could generalize across hundreds of tasks.

### Two-phase framework

| Phase | Data | Objective | Scale |
|---|---|---|---|
| **Pre-training** | Massive unlabeled data (web, books, code) | Self-supervised loss (next-token, masked, contrastive) | Billions of parameters, trillions of tokens |
| **Adaptation** | Small labeled downstream data (or none) | Task-specific loss or prompt | Thousands of examples |

### Adaptation modes (by data size)

| Mode | # Labeled examples | Method |
|---|---|---|
| **Zero-shot** | 0 | Prompt only — no parameter update |
| **Few-shot** | 5–50 | In-context examples in prompt, or fine-tune |
| **Fine-tuning** | 100–100k | Update parameters on labeled data |

**Historical shift:** GPT-3 (2020) showed zero-shot generalization at scale. By 2022–2026, zero-shot / prompting is the default; collecting labeled downstream data is rare.

### The key insight

Fine-tuning and linear probing use the **same loss function** as training from scratch — the only difference is **initialization**. Starting from a pre-trained θ̂ finds a better local optimum (lower test error) than random initialization, even though both might reach zero training loss. This implies the loss landscape has many global minima with very different generalization properties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Visualization: multiple global minima with different test performance ---
# Toy 1D illustration of loss landscape
x = np.linspace(-3, 4, 400)

# Training loss: multiple global minima at 0
def train_loss(x):
    return 0.3 * (np.sin(2*x)**2 * np.cos(x)**2) * np.exp(-0.1*x**2)

# Test loss: only one basin generalizes well (near pretrained init)
def test_loss(x):
    return 0.5 + 0.4*np.exp(-2*(x-0.5)**2) - 0.3*np.exp(-3*(x+1.5)**2) + 0.2*np.sin(x)

train_l = train_loss(x)
test_l  = test_loss(x) - test_loss(x).min() + 0.02  # shift for clarity

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, test_l, color='steelblue', lw=2, label='Test loss (generalization)')
ax.axvline(x=0.5,  color='green',  ls='--', lw=1.5, label='Pre-trained init → good basin')
ax.axvline(x=-1.8, color='red',    ls='--', lw=1.5, label='Random init → poor basin')
ax.axvline(x=2.5,  color='orange', ls='--', lw=1.5, label='Another random init')
ax.set_xlabel('Parameter space (1D projection)')
ax.set_ylabel('Loss')
ax.set_title('Foundation models: pre-trained init finds better generalization basin')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Key insight: same loss function, same training data — only initialization differs.')
print('Pre-trained init consistently finds basins with lower test error.')

## 3. Representation Learning

**Goal:** learn a function $f_\theta: \mathcal{X} \to \mathbb{R}^m$ that maps raw data to a dense vector (representation/embedding/feature) useful for many tasks.

The representation $z = f_\theta(x) \in \mathbb{R}^m$ should capture semantic information linearly — so that downstream tasks can be solved with a simple linear head.

### Pre-training objectives for representations

| Method | Loss | Used in |
|---|---|---|
| **Contrastive (SimCLR, CLIP)** | Attract positives, repel negatives | Vision, vision-language |
| **Masked prediction (BERT, MAE)** | Predict masked tokens/patches from context | Language, vision |
| **Next-token prediction (GPT)** | Predict next token — generative | Language |
| **Diffusion denoising** | Predict noise ε — generative | Images, video |

Today's lecture focuses on the **adaptation** side — how to use any of these representations for downstream tasks.

## 4. Linear Probing

**Setup:** given a pre-trained encoder $f_{\hat\theta}: \mathcal{X} \to \mathbb{R}^m$ with frozen weights $\hat\theta$, learn a linear head $w \in \mathbb{R}^m$ for a downstream task.

**Prediction:** $\hat{y} = w^\top f_{\hat\theta}(x)$ (regression) or $\hat{y} = \text{softmax}(W f_{\hat\theta}(x))$ (classification)

**Training:** minimize over $w$ only:
$$\min_w \frac{1}{n}\sum_{i=1}^n \ell\!\left(w^\top f_{\hat\theta}(x^{(i)}),\, y^{(i)}\right)$$

Since $f_{\hat\theta}$ is frozen, this is a standard linear model — convex in $w$, closed-form solution.

**Why it works:** good representations make the task linearly separable. The pre-training loss encourages $f_\theta$ to organize the embedding space so that semantically similar inputs cluster together.

**Modern use:** even for LLMs, linear probing is used for mechanistic interpretability — e.g., fitting a linear probe to internal activations to detect whether a concept is linearly encoded at a given layer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# --- Linear probing demo on digits ---
# Simulate two "backbones": random projection vs PCA (better representation)
digits = load_digits()
X, y   = digits.data.astype(float), digits.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

# Scale raw features
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)

results = {}

# Backbone 1: Random projection (poor representation)
np.random.seed(42)
R = np.random.randn(64, 20) / np.sqrt(64)     # random 64 → 20
Z_tr_rand = X_tr_s @ R
Z_te_rand = X_te_s @ R
clf_rand = LogisticRegression(max_iter=1000, C=1.0).fit(Z_tr_rand, y_train)
results['Random backbone'] = clf_rand.score(Z_te_rand, y_test)

# Backbone 2: PCA (structured representation)
pca = PCA(n_components=20, random_state=0).fit(X_tr_s)
Z_tr_pca = pca.transform(X_tr_s)
Z_te_pca = pca.transform(X_te_s)
clf_pca  = LogisticRegression(max_iter=1000, C=1.0).fit(Z_tr_pca, y_train)
results['PCA backbone'] = clf_pca.score(Z_te_pca, y_test)

# Backbone 3: Raw pixels (no backbone, direct linear)
clf_raw = LogisticRegression(max_iter=1000, C=1.0).fit(X_tr_s, y_train)
results['No backbone (raw)'] = clf_raw.score(X_te_s, y_test)

print('Linear probing accuracy (10-class digits):')
for name, acc in results.items():
    print(f'  {name:<25}: {acc*100:.1f}%')

# Visualize embeddings (PCA → 2D for plotting)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for ax, (Z_tr, title) in zip(axes, [(Z_tr_rand[:, :2], 'Random backbone (2D)'),
                                      (Z_tr_pca[:, :2], 'PCA backbone (2D)')]):
    for cls in range(10):
        mask = y_train == cls
        ax.scatter(Z_tr[mask, 0], Z_tr[mask, 1], c=[colors[cls]], s=8, alpha=0.5, label=str(cls))
    ax.set_title(title); ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
    ax.legend(title='Digit', fontsize=7, ncol=2, loc='upper right')
    ax.grid(alpha=0.2)

plt.suptitle('Representation Quality Determines Linear Probing Performance', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Fine-Tuning — Warm Initialization

**Setup:** same downstream loss as linear probing, but now optimize **both** $\theta$ and $w$:

$$\min_{w, \theta} \frac{1}{n}\sum_{i=1}^n \ell\!\left(w^\top f_\theta(x^{(i)}),\, y^{(i)}\right), \quad \theta \text{ initialized to } \hat\theta$$

The loss function is identical to training from scratch. The only change: $\theta$ starts from the pre-trained $\hat\theta$ instead of random.

**Multiple global minima phenomenon:**
- Overparameterized neural networks have many solutions with zero training loss
- Not all are equal in test performance (generalization)
- Pre-trained initialization consistently lands in a basin with better generalization
- This is an active area of theoretical research (implicit regularization, flat minima)

**Practical notes:**
- Use a smaller learning rate for fine-tuning than pre-training (don't destroy representations)
- Often freeze early layers, fine-tune later layers
- Risk of **catastrophic forgetting**: fine-tuning on one task degrades performance on others

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Compare: scratch vs fine-tune vs linear probe on limited data ---
# Simulate with a 2-layer MLP; "pre-trained" backbone = PCA features as stand-in

from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X, y   = digits.data.astype(float), digits.target
scaler = StandardScaler()

# Full pre-training simulation: PCA on all data (acts as "pre-trained backbone")
pca_full = PCA(n_components=32, random_state=0).fit(scaler.fit_transform(X))

n_downstream_sizes = [20, 50, 100, 200, 500]
acc_scratch, acc_lp, acc_ft = [], [], []

X_te_all, y_te_all = None, None

for n_down in n_downstream_sizes:
    idx_all  = np.random.RandomState(0).permutation(len(y))
    idx_down = idx_all[:n_down]
    idx_test = idx_all[-400:]

    X_tr, y_tr = X[idx_down], y[idx_down]
    X_te, y_te = X[idx_test],  y[idx_test]

    sc = StandardScaler().fit(X_tr)
    X_tr_s, X_te_s = sc.transform(X_tr), sc.transform(X_te)

    # (1) Scratch: MLP trained directly
    mlp = MLPClassifier((64, 32), max_iter=500, random_state=0).fit(X_tr_s, y_tr)
    acc_scratch.append(mlp.score(X_te_s, y_te))

    # (2) Linear probe on PCA backbone
    Z_tr = pca_full.transform(sc.transform(X_tr))
    Z_te = pca_full.transform(sc.transform(X_te))
    clf  = LogisticRegression(max_iter=500, C=10.0).fit(Z_tr, y_tr)
    acc_lp.append(clf.score(Z_te, y_te))

    # (3) Fine-tune: MLP initialized from PCA-warmed weights (simulated by warm init)
    # Use PCA features as input to a small MLP (simulates warm-start backbone)
    mlp_ft = MLPClassifier((32, 16), max_iter=500, random_state=0).fit(Z_tr, y_tr)
    acc_ft.append(mlp_ft.score(Z_te, y_te))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_downstream_sizes, [a*100 for a in acc_scratch], 'o-', color='red',      label='Train from scratch')
ax.plot(n_downstream_sizes, [a*100 for a in acc_lp],      's-', color='steelblue',label='Linear probe')
ax.plot(n_downstream_sizes, [a*100 for a in acc_ft],      '^-', color='green',    label='Fine-tune (warm init)')
ax.set_xlabel('# downstream labeled examples')
ax.set_ylabel('Test accuracy (%)')
ax.set_title('Pre-trained init vs Scratch — Limited Data Regime')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('With few labels, pre-trained initialization is far better than scratch.')

## 6. LPFT — Linear Probe then Fine-Tune

**Motivation:** When fine-tuning jointly from the start, a randomly initialized $w$ can distort the pre-trained backbone $\hat\theta$ early in training — the random head sends conflicting gradients into the representation layers.

**LPFT algorithm:**

**Step 1 — Linear Probe:** fix $\theta = \hat\theta$, find good head:
$$w_0 = \arg\min_w \mathcal{L}(w, \hat\theta)$$

**Step 2 — Fine-Tune:** starting from $(w_0, \hat\theta)$, jointly optimize:
$$\min_{w, \theta} \mathcal{L}(w, \theta) \quad \text{initialized at } (w_0, \hat\theta)$$

**Why it works:** 
- LP step aligns $w_0$ with the pre-trained feature space before any backbone update
- Subsequent fine-tuning nudges the backbone slightly, rather than destroying it
- The change to $\theta$ is smaller when $w_0$ is already compatible

**Current status:** LPFT is the dominant approach when labeled downstream data is available and full fine-tuning is needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Conceptual visualization of LPFT training dynamics ---
# Show how LP → FT avoids the early instability of joint fine-tuning from random w

steps = np.arange(200)
np.random.seed(3)

# Simulated training loss curves
def smooth(arr, w=10):
    return np.convolve(arr, np.ones(w)/w, mode='same')

# Joint FT from random w: high early instability
noise_jt = np.random.randn(200) * 0.08
ft_curve  = smooth(1.2 * np.exp(-0.02*steps) + 0.15 + noise_jt)

# LPFT: LP stage (fast convergence with frozen backbone), then FT
lp_stage  = smooth(0.9 * np.exp(-0.05*steps[:80]) + 0.2 + np.random.randn(80)*0.05)
ft_stage  = smooth(0.35 * np.exp(-0.04*steps[:120]) + 0.1 + np.random.randn(120)*0.03)
lpft_curve = np.concatenate([lp_stage, ft_stage])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(steps, ft_curve,   color='red',      lw=1.5, label='Fine-tune (random w init)')
axes[0].plot(steps, lpft_curve, color='steelblue', lw=1.5, label='LPFT')
axes[0].axvline(80, color='gray', ls=':', lw=1, label='LP → FT switch')
axes[0].set_xlabel('Training steps'); axes[0].set_ylabel('Training loss')
axes[0].set_title('LPFT: Stable Training Dynamics')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3); axes[0].set_ylim(0)

# Representation drift: how much θ changes from θ̂
drift_ft   = smooth(0.3*(1 - np.exp(-0.05*steps)) + np.random.randn(200)*0.02)
drift_lpft = smooth(np.concatenate([np.zeros(80), 0.1*(1-np.exp(-0.1*steps[:120]))])
                    + np.random.randn(200)*0.01)

axes[1].plot(steps, drift_ft,   color='red',      lw=1.5, label='Fine-tune')
axes[1].plot(steps, drift_lpft, color='steelblue', lw=1.5, label='LPFT')
axes[1].axvline(80, color='gray', ls=':', lw=1, label='LP → FT switch')
axes[1].set_xlabel('Training steps'); axes[1].set_ylabel('||θ - θ̂|| (representation drift)')
axes[1].set_title('LPFT: Less Backbone Distortion')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3); axes[1].set_ylim(0)

plt.suptitle('Linear Probe then Fine-Tune (LPFT)', fontsize=12)
plt.tight_layout()
plt.show()

## 7. LoRA — Low-Rank Adaptation

### Motivation

Full fine-tuning updates every parameter in $\theta$. For a model with $D = 10^9$ parameters:
- Parameter storage: $D$ floats
- Gradient storage: $D$ floats
- Optimizer momentum (Adam): $2D$ floats (1st and 2nd moment)
- Total optimizer state: ~$4D$ — the bottleneck for memory

**Key observation:** When adapting from a powerful pre-trained model, the **change** to $\theta$ is small. We don't need all $D$ degrees of freedom to represent a small update.

### LoRA formulation

For each weight matrix $W_0 \in \mathbb{R}^{d \times d}$, instead of learning $W = W_0 + \Delta W$ with $\Delta W \in \mathbb{R}^{d \times d}$:

$$W = W_0 + \underbrace{A \cdot B}_{\text{low-rank update}}, \quad A \in \mathbb{R}^{d \times r},\; B \in \mathbb{R}^{r \times d},\; r \ll d$$

**Initialization:** $A$ initialized to zero (so $W = W_0$ at start), $B$ initialized randomly.

**Optimization:** freeze $W_0$, learn only $A$ and $B$:
$$\min_{A, B} \mathcal{L}(W_0 + A B, \text{data})$$

### Parameter count

| Method | Parameters to optimize | Example ($d=1000$, $r=10$) |
|---|---|---|
| Full fine-tune | $d^2$ | $10^6$ |
| LoRA | $2dr$ | $20{,}000$ (50× fewer) |

### What LoRA actually saves

- **Optimizer state:** gradient + momentum only for $A$, $B$ — not for frozen $W_0$
- **Compute:** forward pass still needs $W_0 x$ (no saving); backward through $A, B$ is cheaper
- **Memory (key benefit):** with Adam, full fine-tune needs ~$12D$ floats for state; LoRA needs state only for $A, B$ ($\sim 4 \times 2dr$ per layer)

### Multi-user serving (the killer use case)

LoRA enables serving thousands of custom fine-tuned models from one GPU cluster:
- $W_0$ (base model) shared across all users — loaded once
- Each user has their own small $(A_i, B_i)$ adapter — swapped in per request
- This is exactly how OpenAI/Google offer fine-tuning APIs — only LoRA updates, not full weights

### LoRA is a structural bias

$\Delta W = AB$ has rank $\leq r$. This restricts the space of possible updates — a form of regularization. When $r$ is small, only low-rank adaptations are possible, which may not capture all task-specific changes. Choosing $r$ is a hyperparameter (typical values: 4, 8, 16, 64).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- LoRA: verify forward pass equivalence and parameter count ---

np.random.seed(42)
d     = 64    # matrix dimension
r_list = [1, 2, 4, 8, 16, 32, 64]

# Base weight matrix (pre-trained)
W0 = np.random.randn(d, d) / np.sqrt(d)

# True delta W: a random low-rank matrix of rank 4
true_rank = 4
A_true = np.random.randn(d, true_rank) / np.sqrt(true_rank)
B_true = np.random.randn(true_rank, d) / np.sqrt(true_rank)
delta_W_true = A_true @ B_true    # rank-4 update
W_target = W0 + delta_W_true

# For each rank r, fit LoRA via SVD of delta_W_true
U, S, Vt = np.linalg.svd(delta_W_true)
recon_errors = []
for r in r_list:
    delta_W_r = (U[:, :r] * S[:r]) @ Vt[:r, :]
    err = np.linalg.norm(delta_W_true - delta_W_r, 'fro') / np.linalg.norm(delta_W_true, 'fro')
    recon_errors.append(err)

# Parameter count comparison
param_full = d * d
param_lora = {r: 2 * d * r for r in r_list}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Singular value spectrum of the true update
axes[0].bar(range(1, d+1), S, color='steelblue', alpha=0.8)
axes[0].axvline(true_rank + 0.5, color='red', ls='--', lw=2, label=f'True rank = {true_rank}')
axes[0].set_xlabel('Singular value index'); axes[0].set_ylabel('Singular value')
axes[0].set_title('ΔW singular spectrum (low-rank update)')
axes[0].set_xlim(0, 20); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Reconstruction error vs rank
axes[1].semilogy(r_list, recon_errors, 'o-', color='red', ms=6)
axes[1].axvline(true_rank, color='gray', ls='--', lw=1, label=f'True rank')
axes[1].set_xlabel('LoRA rank r'); axes[1].set_ylabel('Relative Frobenius error')
axes[1].set_title('LoRA Approximation Error vs Rank')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Parameter count
axes[2].bar(['Full\nfine-tune'] + [f'LoRA\nr={r}' for r in r_list],
            [param_full] + [param_lora[r] for r in r_list],
            color=['red'] + ['steelblue']*len(r_list), alpha=0.8)
axes[2].set_ylabel('# parameters to optimize')
axes[2].set_title(f'Parameter Count (d={d})')
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('LoRA: Low-Rank Adaptation Analysis', fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nFull fine-tune: {param_full:,} parameters')
for r in [1, 4, 16]:
    print(f'LoRA r={r:2d}:      {param_lora[r]:,} parameters  ({100*param_lora[r]/param_full:.1f}% of full)')

print('\nReconstruction of rank-4 true update:')
for r, err in zip(r_list, recon_errors):
    print(f'  r={r:2d}: {err:.6f} relative error')

In [ ]:
import numpy as np

# --- LoRA forward pass: verify W0 + AB gives same result as W_target ---
np.random.seed(0)
d, r = 32, 4
W0 = np.random.randn(d, d) / np.sqrt(d)

# LoRA parameters (after training, suppose we found good A, B)
A_lora = np.random.randn(d, r) / np.sqrt(r)
B_lora = np.random.randn(r, d) / np.sqrt(r)

# Two equivalent forward passes for input x
x = np.random.randn(d)

# Option 1: merged weight (at inference, merge W0 + AB once)
W_merged = W0 + A_lora @ B_lora
out_merged = W_merged @ x

# Option 2: separated (W0 frozen, add low-rank correction)
out_lora = W0 @ x + A_lora @ (B_lora @ x)

print('LoRA forward pass correctness:')
print(f'  Max difference between merged and separated: {np.max(np.abs(out_merged - out_lora)):.2e}')
print('  → Identical outputs (as expected)')

print('\nMemory breakdown for d=1024, r=8, L=32 transformer layers:')
d_large, r_small, L = 1024, 8, 32
W0_params     = d_large * d_large * L
lora_params   = 2 * d_large * r_small * L
# Adam optimizer: 2x for moment estimates, only for LoRA params
adam_full  = 3 * W0_params          # weights + 2 moments
adam_lora  = W0_params + 3 * lora_params  # frozen W0 + LoRA with moments
print(f'  W0 params (frozen):        {W0_params:>12,}')
print(f'  LoRA params (A+B):         {lora_params:>12,}  ({100*lora_params/W0_params:.1f}% of W0)')
print(f'  Adam state — full FT:      {adam_full:>12,}')
print(f'  Adam state — LoRA:         {adam_lora:>12,}  ({100*adam_lora/adam_full:.1f}% of full)')

## Summary

| Method | Optimize | # Params | When to use |
|---|---|---|---|
| **Linear probe** | $w$ only | $m \cdot C$ | Many downstream tasks, small data |
| **Fine-tune** | $w + \theta$ | Full model | Sufficient data, one task |
| **LPFT** | $w$ → $(w, \theta)$ | Full model | Best of both — current standard |
| **LoRA** | $A, B$ only | $2dr$ per layer | Resource-limited; multi-user serving |

### Connection to lecture arc

```
Data distribution p_data
      ↓ forward process (fixed)
Noisy samples x_t
      ↓ train ε_θ to predict noise
Foundation model (pre-trained θ)
      ↓ adaptation
      ├── Linear probe: frozen θ, train head w
      ├── Fine-tune:    warm-start θ, train both
      ├── LPFT:         LP first, then FT
      └── LoRA:         frozen θ, train low-rank A,B
```

**Next lecture (L13):** pre-training objectives for language — next-token prediction, cross-entropy loss, perplexity, scaling laws.